# Vanilla Transformer

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Positional Encoding

One of the most important features of Transformers is **parallelism:** these models can process an entire sentence all at once, making them incredibly fast. However, this has a major side effect: the model becomes **permutation invariant**.
This means that the model treats the sentence "The cat chases the mouse" exactly the same as "The mouse chases the cat".
The model has no sense of order.

To overcome this problem, we add a **unique signature** to each word's vector, representing its position.
To do so, we use **sine and cosine functions**, so that each position gets a unique fingerprint and the model can learn to attend the relative positions.

This signature is computed in a **Positional Encoder**, placed at the beginning of the model, following the formulas:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$


$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

where:

*   pos: the position of the word in the sentence (e.g., 0 for the first word, 1 for the second...).
*   i: the index of the dimension within the embedding vector.
*   2i: even dimensions.
*   2i + 1: odd dimensions.
*   $d_{model}$: the total depth/dimension of the embedding.



In [31]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # Define a dropout layer to randomly turn off part of the resulting signal
        self.dropout = None # TODO

        # Define positional encoding matrix
        pe = None # TODO: initialize a torch tensor to zero. Dim: max_len x d_model

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Encode even dimensions
        pe[:, 0::2] = None  # TODO: apply sin function to position * div_term
        pe[:, 1::2] = None  # TODO: apply cos function

        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # self.pe[:x.size(0), :] returns positional encodings from pos 0 up to current sequence length
        x = None  # TODO: sum up token embeddings (x) and position embeddings

        return None   # TODO: apply dropout and return the result

## Dot Product Attention Mechanism

Once the model has a sense of order thanks to the Positional Encoding, it needs a way to look at the words and decide which ones are **important relative to each other.**

Every word in a sequence is transformed into three different vectors:


*   **Query (Q):** the current word searching for context.
*   **Key (K):** the label other words use to find this word.
*   **Value (V):** the content we want to extract.

The mechanism is defined by the following equation:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$





In [32]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, query, key, value, mask=None):
        d_k = query.size(-1)

        # Compute similarity scores scaled by d_k
        scores = None  # TODO: compute matrix multiplication between query and key.transposed(-2, -1), and scale by sqrt of d_k

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Apply softmax to scores
        p_attn = None # TODO

        # TODO: return matrix multiplication between p_attn and value, and p_attn
        return None, None

## Multi-Head Attention Mechanism

Transformers use a **multiple parallel attention mechanism**, composed of multiple parallel "heads", to capture **different perspectives** simultaneously (e.g., head 1 focuses on contextual meaning, head 2 on grammar, head 3 looks for the subject of the sentence...).

In [33]:
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % h == 0

        self.d_k = d_model // h
        self.h = h  # Number of attention heads

        # Define the query
        self.query = None # TODO: linear layer with shape (d_model, d_model)
        self.key = None # TODO: linear layer as above
        self.value = None # TODO: linear layer as above
        self.output = None  # TODO: linear layer as above

        self.attn = None  # TODO: instantiate a scaled dot product attention layer

        self.dropout = None # TODO: define a dropout layer

    def forward(self, query, key, value, mask=None):
        nbatches = query.size(0)

        # 1) Linear projections and reshape for multi-head
        query = self.query(query).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
        key = self.key(key).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
        value = self.value(value).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)

        # 2) Apply attention on all the projected vectors in batch
        x, attn = None  # TODO

        # 3) "Concat" using a view and apply a final linear
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        x = None  # TODO: apply final output layer
        return x, attn

## Feed Forward Layer & Normalization Layer

After the Multi-Head Attention we put a **Feed-Forward layer** to process **what the words actually mean.**

The **Layer Normalization** is then used to stabilize training, preventing internal covariate shift.

In [34]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()

        self.w_1 = None # TODO: define a linear layer (d_model, d_ff)
        self.w_2 = None # TODO: define a second linear layer (d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return None # TODO: w_1 -> relu -> dropout -> w_2

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

## Sublayer Connection

The **Sublayer Connection** implements two critical concepts: **Residual Connections** and **Layer Normalization**.

*   **Residual Connection:** in deep neural networks, during backpropagation, gradients get multiplied by small numbers over and over, vanishing to zero as they reach early layers. The residual connection allows the gradient to flow through **preserving the original information.**
*   **Layer Normalization:** ensures input have a consistent mean and variance, **preventing exploding values** that could crash training.



In [35]:
class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = None # TODO: instantiate a Layer Normalization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x))[0])

## Encoder Layer

The primary goal of the **Encoder Layer** is to take a set of word embeddings and **re-write** them based on their surroundings. The output will be a sequence of vectors where each vector contains "context".

Transformers are able to build increasingly complex representations of the input text by **stacking multiple copies of the Encoder Layer** on the top of each other.

In [36]:
class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = None     # TODO
        self.feed_forward = None  # TODO
        self.sublayer = None  # TODO: a nn.ModuleList populated with 2 Sublayer Connections
        self.size = None  # TODO

    def forward(self, x, mask):
        # TODO: apply the first sublayer (remember self.sublayer can be treated as an array) to
        # x and the result of the attention mechanism. To retrieve the result of the self.attn, you can
        # use the following lambda notation directly as second parameter of the sublayer
        # lambda: x: self.self_attn(x, x, x, mask)
        # Notice that x here is query, key and value, because this is a self-attention mechanism
        x = None
        return None # TODO: apply the second sublayer to x and self.feed_forward

# Stack of Encoder Layers
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super(Encoder, self).__init__()

        self.layers = None  # TODO: ModuleList populated with N layers
        self.norm = None  # TODO: instantiate a LayerNorm. As size you can use layer.size

    def forward(self, x, mask):
        for layer in self.layers:
            pass  # TODO: pass x through each layer. Remember to pass also the mask
        return self.norm(x)

## Decoder Layer

The goal of the **Decoder Layer** is to write the output.
Its architecture looks very similar to the Encoder, with two major differences:



*   **Masked Self-Attention:** in the Encoder, every word can see every other word, but in the Decoder we must block words that come after the current one. Predictions can be made based **only on the past**, and not the future.
*   **Cross Attention:** a third sublayer is added, to allow the Decoder to look back to the original input to decide which words are most relevant to the output. In this layer, the Queries come from the Decoder, while Keys and Values come from the Encoder.





In [37]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = None  # TODO: ModuleList of 3 Sublayer Connections

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory ## This is the encoder output (memory)

        # TODO: apply first sublayer to x. Second argument can be expressed using the same lambda notation
        # as before. Use tgt_mask as mask.
        x = None

        # TODO: apply second sublayer. Second argument can be expressed through lambda notation.
        # x will be the query, and m will be both key and value.
        # Use src_mask as mask
        x = None

        # Last sublayer
        return self.sublayer[2](x, self.feed_forward)

class Decoder(nn.Module):
    def __init__(self, layer, N):
        super(Decoder, self).__init__()

        self.layers = None # TODO: ModuleList of N layers
        self.norm = None  # TODO: instantiate Layer Normalization

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
          pass # TODO: apply layer to x, remember to pass also memory, src_mask and tgt_mask
        return self.norm(x)

## Transformer Block

The **Transformer block** combines the encoder, decoder, and input/output embeddings.
It takes the source and target sequences and their masks as input.

In [38]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(Transformer, self).__init__()
        self.encoder = None     # TODO
        self.decoder = None     # TODO
        self.src_embed = None   # TODO
        self.tgt_embed = None   # TODO
        self.generator = None   # TODO

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = None   # TODO: apply encoding function to src and src_mask
        decoded = None  # TODO: apply decoding function to memory, src_mask, tgt and tgt_mask
        return None     # TODO apply generator

    def encode(self, src, src_mask):
        return None   # TODO: apply encoder to self.src_embed(src) and src_mask

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return None   # TODO: apply decoder

## Generator

The **generator** takes the final output of the decoder and **projects** it to the vocabulary size.
Apply a log-softmax to get probability distributions over the target vocabulary.

In [39]:
class Generator(nn.Module):
    def __init__(self, d_model, vocab_size):
        super(Generator, self).__init__()
        self.proj = None  # TODO: define a linear layer (d_model, vocab_size)

    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)

## Model Definition

In [40]:
import copy

def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1):
    # If we do not use deepcopy, all layers would share the same weights
    c = copy.deepcopy

    attn = None       # TODO: define a Multi-Head Attention block
    ff =   None       # TODO: define a Feed Forward block
    position =  None  # TODO: define a Positional Encoding block

    model = Transformer(
        # TODO: define Encoder block, using c(attn) and c(ff) to have independent weights
        None,
        # TODO: define Decoder block, using c(attn), c(attn) and c(ff) to have independent weights
        None,
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        # TODO: define the Generator
        None)

    # Initialize parameters with Glorot / fan_avg.
    # Good initialization ensures:
    # 1. gradients don't explode
    # 2. gradients don't vanish
    # 3. training starts smoothly
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

### Input Embeddings

In [41]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

### Example Usage

In [42]:
# Define dummy vocab size
src_vocab_size = 10000
tgt_vocab_size = 8000

# Model's paramters
N = 2 # Number of encoder/decoder layers
d_model = 256
d_ff = 1024
h = 4
dropout = 0.1

# Create an instance of the Transformer model
model = None  # TODO

# Create some random input tensors (source and target sequences) and their corresponding masks.
# The source mask typically handles padding, while the target mask prevents the decoder from
# "seeing" future tokens during training.
batch_size = 4
src_seq_len = 40
tgt_seq_len = 25

src = torch.randint(0, src_vocab_size, (batch_size, src_seq_len))
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_seq_len))
src_mask = torch.ones(batch_size, 1, src_seq_len)
tgt_mask = torch.ones(batch_size, tgt_seq_len, tgt_seq_len).tril(diagonal=0).unsqueeze(1)

# Perform a forward pass through the model with the dummy data and masks.
# Print the output shape to verify the model's output dimensions.
# TODO: Perform a forward pass
output = None


print("Output shape:", output.shape) # Should be (batch_size, tgt_seq_len, tgt_vocab_size)

print("Transformer model created and a forward pass performed with dummy data.")

Output shape: torch.Size([4, 25, 256])
Transformer model created and a forward pass performed with dummy data.
